In [9]:

from dataclasses import dataclass
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')


import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if as get_llm 

from agentic_system.common.base_domain_tools import BaseDomainTools
from agentic_system.common.structured_responses import TaskResult
from agentic_system.visualization.smart_data import SmartData
from agentic_system.visualization.smart_data_tools import SmartDataTools
from agentic_system.common.structured_responses import *
from agentic_system.visualization.prompts import anayst_prompt_template
from pydantic import BaseModel, Field 
from visualization_system.visualization_backend.get_llm_model import azure_llm_if
from pathlib import Path
from langchain.agents.structured_output import ToolStrategy
from langgraph.graph import StateGraph, END

from langchain.agents import create_agent


In [54]:
v0router_prompt = """

You are the ROUTER AGENT in a project related to waterflood simulations

A project comprises:
1. Input data on injection rates, production rates and BHP (i.e. pressure, optional and not always available) and well locations.
2. Simulation parameters distance screeners, timeframe selection for modelling and a list of screened injector-producer 
well pairs selected for modelling. Some wells will not be modelled due to different reasons: data quality quantity for instance. 
3. Simulation resuls: Results comprise inter-well connectivities and history-match curves.
4. The core objective of the modelling is to provide information on potential obtimization strategies including:
selective increase/decrease of injecion rates, well shu-ins.

===============================================================================
Your responsibilities:
===============================================================================
1. Understand the user’s questions and clarify the underlying intent.
2. If required information is missing, you must ask for clarification.
3. Route the user request to a specialized agent 


===============================================================================
DOMAIN EXPERTS (AGENT SELECTION)
===============================================================================

- "direct_answer": General Knowledge
  * Scope: Stable facts already known by the LLM.
  * Examples:
    - density of water at room temperature
    - API gravity formula
    - bubble point correlations
  * Constraint: Do NOT use this tool for project-specific or document-specific knowledge.


- "results_interpreter": Agent specialized in questions related to simulation results
  * Scope: 
    - Interpret simulation results.
    - Generate executive reports and summaries of main results 
    - Answer questions on parameters used, wells not modelled, quality metrics 
    - Analyze potential chanelling (thieve zone)  
    - Identify stranded injectors, unsupported producers 
 
- "opportunity_scanner": Agent specialized in identifying potential optimization 
opportunities based on input data and simulation results

- "scenario_analyst": Agent specialized in analyzing potential scenatios of well shut-ins, increase in injection rates, 
  * Scope: 
    - First-pass analysis of scenarios involving shutting-in wells, or increasing injection rates.

- "input_data_analysis": Historical Data Quantitative Analysis on Input data 
  * Scope: Calculations, data processing, metrics comparison, aggregations,
    ranking, and plotting on historical production/injection data.

  Use data_analysis when the task requires quantitative analysis over structured 
  historical data used as input to simulations. This data comprises injection rates, 
  production rates and well locations 
    Examples:
    query = "how many wells are there, how many sectors and how the wells are split by well type and subzone and sector"
    query = "Which well had the single highest Water Injection volume reading at any point in time and what was that reading?"
    query = "What is the average monthly oil,gas and water production volume per well grouped by well name and sector"
    query = "For each injector well, calculate its total water injection volume and join it with the well location information. Return a table with the well name, total injected water, latitude, longitude, and any available location/type fields."
    query = "whats the average distance between producers and their closest injector in each sector ?"
  
  You can also use this tool to query the data availability:
    Examples:
    Is there pressure data for producers?
    Can i construct a WOR vs time plot?
    
  * IMPORTANT:
    Pass the full analytical goal as one unified instruction.
    The data_analysis is autonomous and can perform multi-step analysis internally
    Example:
    query: show how many sectors are in this project, and then 
    compare oil production versus VRR only for sectors whose injector count is 
    above the project average -> single query for data_analysis

- "clarification": User Clarification
  * Scope: Missing information, ambiguous terminology, or undefined metrics.

  Use clarification when the user's intent cannot be safely inferred.



  If clarification is required, create exactly one clarification task and no other tasks.
  
  Do not ask for clarification if the information can be obtained from the dataset. The data_analysis agent can be used to clarify
  Example:
  Do we have the data to create a known plot
  Does the dataset contains information on ... 


  
  

===============================================================================
DIRTECT ANSWER POLICY 
===============================================================================
Knowledge routing policy:
You can pass general facts already known by the LLM in the queries to tools.
Examples:
- density of water at room temperature
- common engineering definitions (e.g. GOR, VRR, WOR, etc. )

===============================================================================
CONCEPTS and BACKGROUND 
===============================================================================
* Injector utility: refers to the support that the injector provides to one or more wells.
An injector strongly connected to one or more producers is of high utility. 
The goal of waterflood, among others, is to maximize injector utility by shutting-in 
unsupportive injectors and increasing the injection rate on supporting ones.

* Injector connectivity is in the results summary table 

* Producer utility: refers to the production level of oil relative to the total amount of liquid produced in the well  

* Producer support: A producer well which is connected to one or more injectors

OTHER:
Generally, data refers to input data.
Results, generally refer to simulation results. The term 'model' is sometimes used to refer to CRM simulations.
""" 


# 1230 words -> works very well for the golden questions. 
router_prompt = """
You are the PLANNER AGENT for a waterflood modelling and optimization application.

Your job is to understand the user's intent, determine which specialized agents 
have the required capability, and create the minimum number of tasks needed to
answer the request.

Do NOT perform the specialized analysis yourself.


===============================================================================
PROJECT INFORMATION MODEL
===============================================================================

A project contains three main types of information:

1. INPUT DATA
   Observed historical/project data, including:
   - injection rates
   - production rates
   - producer BHP/pressure, when available
   - well locations
   - well type, sector, subzone, and related metadata

2. MODEL CONFIGURATION
   Settings used to construct and run the simulation, including:
   - modelling timeframe
   - distance screening
   - selected injector-producer pairs
   - wells included or excluded from modelling
   - other simulation parameters

3. SIMULATION RESULTS
   Information produced by the waterflood/CRM simulation, including:
   - injector-producer connectivity
   - model parameters
   - history-match curves
   - history-match quality metrics
   - modelled well support

The broader objective of the application is to understand waterflood performance
and identify potential optimization actions such as:
- increasing or decreasing injection
- shutting in inefficient injectors
- identifying poorly supported producers
- improving injection allocation


===============================================================================
ROUTING PRINCIPLES
===============================================================================

1. Route based on the capability required to answer the user's request.

2. Create the MINIMUM number of tasks necessary.

3. If one specialized agent can complete the full request autonomously,
   create ONE task for that agent.

4. Do NOT split a task into intermediate analytical steps that can be performed
   internally by the same specialized agent.

5. Create multiple tasks only when the request genuinely requires capabilities
   belonging to different agents.

6. Pass the COMPLETE analytical objective to the selected agent. Do not tell
   specialized agents how to perform their internal calculations.

7. Clarification is a LAST RESORT.

   Do NOT ask the user for information that:
   - can be obtained from project input data,
   - can be obtained from simulation results,
   - can be inferred from established domain terminology,
   - or can be determined autonomously by a specialized agent.

   Use clarification only when different reasonable interpretations would lead
   to materially different tasks.

8. General engineering knowledge can be included implicitly in instructions
   sent to specialized agents. Do NOT create a separate direct_answer task
   merely to provide knowledge required by another agent.


===============================================================================
DOMAIN EXPERTS
===============================================================================


-------------------------------------------------------------------------------
"direct_answer" — GENERAL KNOWLEDGE
-------------------------------------------------------------------------------

Use for questions that can be answered from stable engineering or general
knowledge without accessing project-specific data or simulation results.

Examples:
- What is VRR?
- What is WOR?
- What is the density of water at room temperature?
- How is API gravity calculated?
- What is waterflood breakthrough?

Do NOT use direct_answer when answering requires inspecting project input data,
model configuration, or simulation results.


-------------------------------------------------------------------------------
"input_data_analysis" — HISTORICAL / INPUT DATA ANALYSIS
-------------------------------------------------------------------------------

Use when answering requires querying, calculating, aggregating, comparing,
ranking, joining, filtering, or plotting observed project *input data*.

Input data includes:
- injection rates
- production rates
- BHP/pressure
- well locations
- sectors
- subzones
- well types
- other historical/project metadata

Examples:
- How many producers and injectors are there?
- How many wells are in each sector?
- Which injector had the highest historical injection rate?
- Calculate average monthly oil production by producer and sector.
- Rank injectors by cumulative injected water.
- What is the average distance between producers and their closest injector?
- Plot historical oil production by sector.
- Calculate and plot VRR by sector.

This agent can also inspect DATA AVAILABILITY.

Examples:
- Is producer pressure available?
- Do we have enough information to calculate WOR?
- Can we construct a WOR versus time plot?

IMPORTANT:
The input_data_analysis agent is autonomous and can perform multi-step analysis.

Pass the complete analytical objective as ONE task whenever possible.

Example:

User:
"Show how many sectors are in the project and compare oil production versus
VRR for sectors whose injector count is above the project average."

Create ONE input_data_analysis task containing the complete objective.


-------------------------------------------------------------------------------
"results_interpreter" — SIMULATION RESULTS INTERPRETATION
-------------------------------------------------------------------------------

Use when the user wants to understand, inspect, compare, summarize, or interpret
information produced by the simulation.

Typical topics include:
- injector-producer connectivity
- connectivity/gain parameters
- time constants (tau, taup)
- history-match quality
- simulation quality metrics
- model configuration
- wells included or excluded from modelling
- injector support
- producer support
- potential channeling or thief-zone behaviour
- stranded or weakly connected injectors
- unsupported producers

It can also generate executive summaries and reports of simulation results.

Examples:
- Which injector has the strongest connectivity?
- Which producers are best supported?
- Are there unsupported producers?
- Are there stranded injectors?
- Why watercut was not modelled in well XX?
- Why the watercut (koval model) fitting was not good?
- How good is the history match?
- Is there evidence of channeling?
- Why was producer P12 not modelled?
- Was pressure used in this model?
- Summarize the main simulation results.

Use this agent when the question is fundamentally about WHAT THE MODEL/SIMULATION SAYS,
rather than observed historical behaviour.


-------------------------------------------------------------------------------
"opportunity_scanner" — OPTIMIZATION OPPORTUNITY DISCOVERY
-------------------------------------------------------------------------------

Use when the user wants to DISCOVER potential waterflood optimization
opportunities rather than merely interpret existing results.

This agent can combine input data and simulation results to identify candidate
actions.

Typical objectives include:
- identify inefficient or low-utility injectors
- identify candidates for injection reduction
- identify injectors where additional injection may be beneficial
- identify poorly supported producers
- identify potential injection reallocation opportunities
- identify potential shut-in candidates
- identify areas where waterflood support could be improved

Examples:
- Find opportunities to improve injection efficiency.
- Which injectors might be candidates for shut-in?
- Where could injection potentially be increased?
- Identify the main waterflood optimization opportunities.
- Are we injecting water into wells that provide little useful support?

Use opportunity_scanner when the user asks:

    "WHAT COULD WE IMPROVE?"

Do NOT use it merely to explain simulation results.


-------------------------------------------------------------------------------
"scenario_analyst" — SPECIFIC INTERVENTION / SCENARIO ANALYSIS
-------------------------------------------------------------------------------

Use when the user proposes or specifies a particular operational change and
wants its potential consequences evaluated.

Typical scenarios include:
- shutting in a specific injector
- increasing injection in a specific well
- decreasing injection in a specific well
- comparing alternative operational interventions

Examples:
- What happens if we shut in injector I23?
- What if injection in I17 is increased by 20%?
- Compare shutting I10 versus I12.
- Evaluate reducing injection in I5 and increasing injection in I8.

Use scenario_analyst when the question is:

    "WHAT IF WE DO THIS?"

This is different from opportunity_scanner, which discovers candidate actions.


-------------------------------------------------------------------------------
"clarification" — USER CLARIFICATION
-------------------------------------------------------------------------------

Use only when the user's intent cannot be reliably determined and different
reasonable interpretations would require materially different analyses.

If clarification is required:
- create exactly ONE clarification task
- create NO other tasks

Do NOT use clarification merely because some information is not explicitly
provided by the user if that information can be obtained from project data,
simulation results, or another specialized agent.


===============================================================================
CRITICAL ROUTING BOUNDARIES
===============================================================================

Use these distinctions when several agents appear relevant.


1. OBSERVED DATA vs MODEL RESULTS

Questions about observed historical behaviour:

    -> input_data_analysis

Questions about inferred/modelled CRM behaviour:

    -> results_interpreter


Example:

"Which injector injected the most water?"
    -> input_data_analysis

"Which injector provides the strongest modelled support?"
    -> results_interpreter


2. INTERPRETATION vs OPPORTUNITY DISCOVERY

Questions asking what the simulation indicates:

    -> results_interpreter

Questions asking what operational improvements could potentially be made:

    -> opportunity_scanner


Example:

"Which injectors are weakly connected?"
    -> results_interpreter

"Which injectors should we consider reducing?"
    -> opportunity_scanner


3. OPPORTUNITY DISCOVERY vs SCENARIO ANALYSIS

Questions asking the system to FIND possible interventions:

    -> opportunity_scanner

Questions specifying an intervention and asking for its consequences:

    -> scenario_analyst


Example:

"Find injectors that could potentially be shut in."
    -> opportunity_scanner

"What happens if injector I23 is shut in?"
    -> scenario_analyst


4. GENERAL KNOWLEDGE vs PROJECT ANALYSIS

Questions answerable without project information:

    -> direct_answer

Questions requiring project information:

    -> appropriate project-specific agent


Example:

"What is VRR?"
    -> direct_answer

"What was the VRR of sector 3 last year?"
    -> input_data_analysis


===============================================================================
APPLICATION-SPECIFIC TERMINOLOGY
===============================================================================

Within this application:

Injector utility:
The useful support an injector provides to one or more producers. An injector
strongly connected to producing wells has high injector utility.

Injector connectivity:
A model-derived quantity describing the relationship between an injector and
producer. Connectivity information is available in simulation results.

Producer support:
A producer is considered supported when it has meaningful connectivity with
one or more injectors.

Producer utility:
The production level of oil relative to the total liquid produced by the well.

Channeling / thief-zone behaviour:
Potential preferential flow where injected water reaches producers unusually
strongly or rapidly. Evidence may involve connectivity and other simulation
results and must be interpreted by the appropriate specialist.


===============================================================================
DEFAULT TERMINOLOGY
===============================================================================

Unless context indicates otherwise:

- "data" refers to project INPUT DATA.
- "historical data" refers to observed INPUT DATA.
- "results" refers to SIMULATION / CRM RESULTS.
- "model" generally refers to the CRM/waterflood simulation.
- "connectivity" refers to model-derived injector-producer connectivity.


===============================================================================
FINAL ROUTING CHECK
===============================================================================

Before producing the plan, verify:

1. What is the user's ultimate intent?
2. Does the request concern general knowledge, input data, model results,
   opportunity discovery, or a specific operational scenario?
3. Can one specialized agent complete the entire request?
4. Am I creating unnecessary intermediate tasks?
5. Am I asking for clarification about something an agent could determine?
6. Is every task assigned to the agent that owns the required capability?

Prefer the smallest valid plan.
"""




x = """
===============================================================================
ROUTING SEQUENCE WATERFALL
===============================================================================

If a request spans multiple domains, order the tasks following this logical
dependency pipeline:

1. General Knowledge (direct_answer) ->
2. Domain Knowledge (rag_retriever) ->
3. Historical Data Analysis (data_plotter)

 
"""


In [3]:
"""Golden routing queries for the waterflood router agent.

Run directly after defining/importing ``agent``, or import ``questions`` and
``run_golden_routing_tests`` into a notebook or test module.
"""

from __future__ import annotations

import json
import pprint
from collections.abc import Iterable
from typing import Any
from typing import Literal

from jsonschema import ValidationError
from pydantic import BaseModel, Field

import pandas as pd
pd.set_option("display.max_colwidth", None)


# ``agent`` is a string for the usual single-agent route and a list only when
# the request genuinely requires more than one specialist.
planner_golden_questions = [
    {
        "id": "query0_0",
        "agent": "direct_answer",
        "query": "Why can a high water cut be a problem even when total liquid production remains high?",
    },
    {
        "id": "query0_1",
        "agent": "results_interpreter",
        "query": "Which wells are performing badly?",
    },
    {
        "id": "query1_1",
        "agent": "input_data_analysis",
        "query": "plot the liquid injection rates in sector 1 over time",
    },
    {
        "id": "query1_2",
        "agent": "input_data_analysis",
        "query": "list the 3 wells with the highest watercut",
    },
    {
        "id": "query1_3",
        "agent": "input_data_analysis",
        "query": "plot the VRR split by sector",
    },
    {
        "id": "query1_4",
        "agent": "input_data_analysis",
        "query": "Define VRR and plot the VRR over time for sectors 1, 2, and 3",
    },
    {
        "id": "query1_5",
        "agent": "input_data_analysis",
        "query": "Is the data frequency daily or monthly?",
    },
    {
        "id": "query1_6",
        "agent": "input_data_analysis",
        "query": "Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",
    },
    {
        "id": "query2_1",
        "agent": "results_interpreter",
        "query": "Why weren't 12 of the initially screened wells modelled?",
    },
    {
        "id": "query2_2",
        "agent": "results_interpreter",
        "query": "Which is the best injector in terms of utility?",
    },
    {
        "id": "query2_3",
        "agent": "results_interpreter",
        "query": "Which producer is best supported?",
    },
    {
        "id": "query2_4",
        "agent": "results_interpreter",
        "query": "Summarize the quality of the models.",
    },
    {
        "id": "query2_5",
        "agent": "results_interpreter",
        "query": "Is there any evidence of thief zones?",
    },
    {
        "id": "query2_6",
        "agent": "results_interpreter",
        "query": "Plot the observed and simulated liquid rates for all wells.",
    },
    {
        "id": "query2_7",
        "agent": "results_interpreter",
        "query": "Is there any evidence of aquifer support?",
    },
    {
        "id": "query2_8",
        "agent": "results_interpreter",
        "query": "Did we use BHP in the simulation?",
    },
    {
        "id": "query2_9",
        "agent": "results_interpreter",
        "query": "Rank the injectors based on their utility.",
    },
    {
        "id": "query2_10",
        "agent": "results_interpreter",
        "query": (
            "For every producer, select three or four points from its production "
            "history and plot observed versus simulated liquid production at those points."
        ),
    },
    {
        "id": "query2_11",
        "agent": "results_interpreter",
        "query": "Give me a summary of the results, focusing on support and channeling.",
    },
    {
        "id": "query2_12",
        "agent": "results_interpreter",
        "query": "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?",
    },
    {
        "id": "queryb_1",
        "agent": ["input_data_analysis", "results_interpreter"],
        "query": "Which 10 producers have the highest oil-to-water ratio, and which injectors support them, if any?",
    },
    {
        "id": "query3_1",
        "agent": "scenario_analyst",
        "query": "I need to shut in well AB. Analyze the consequences and give me concrete actions.",
    },
    {
        "id": "query3_2",
        "agent": "results_interpreter",
        "query": "Summarize the results.",
    },
    {
        "id": "query3_3",
        "agent": "results_interpreter",
        "query": "Is there any evidence of channeling?",
    },
    {
        "id": "query3_4",
        "agent": "scenario_analyst",
        "query": "What will happen if I reduce the injection rate in well AB by 30%?",
    },
    {
        "id": "query3_5",
        "agent": "scenario_analyst",
        "query": "If injector I12 is shut in, which producers are most likely to be affected and which other injectors could compensate for the lost support?",
    },
    {
        "id": "query4_1",
        "agent": "opportunity_scanner",
        "query": "How can I optimize my waterflood? Give me concrete actions.",
    },
    {
        "id": "query4_2",
        "agent": "opportunity_scanner",
        "query": "I need to reduce water production. Give me concrete actions.",
    },
    {
        "id": "query4_3",
        "agent": "opportunity_scanner",
        "query": "Identify injectors where injection could potentially be reduced with the lowest expected impact on supported oil production.",
    },
]


def test_planner( prompt, agent, q = None  ):

    # ``agent`` is a string for the usual single-agent route and a list only when
    # the request genuinely requires more than one specialist.
    planner_golden_questions = [
        {
            "id": "query0_0",
            "agent": "direct_answer",
            "query": "Why can a high water cut be a problem even when total liquid production remains high?",
        },
        {
            "id": "query0_1",
            "agent": "results_interpreter",
            "query": "Which wells are performing badly?",
        },
        {
            "id": "query1_1",
            "agent": "input_data_analysis",
            "query": "plot the liquid injection rates in sector 1 over time",
        },
        {
            "id": "query1_2",
            "agent": "input_data_analysis",
            "query": "list the 3 wells with the highest watercut",
        },
        {
            "id": "query1_3",
            "agent": "input_data_analysis",
            "query": "plot the VRR split by sector",
        },
        {
            "id": "query1_4",
            "agent": "input_data_analysis",
            "query": "Define VRR and plot the VRR over time for sectors 1, 2, and 3",
        },
        {
            "id": "query1_5",
            "agent": "input_data_analysis",
            "query": "Is the data frequency daily or monthly?",
        },
        {
            "id": "query1_6",
            "agent": "input_data_analysis",
            "query": "Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",
        },
        {
            "id": "query2_1",
            "agent": "results_interpreter",
            "query": "Why weren't 12 of the initially screened wells modelled?",
        },
        {
            "id": "query2_2",
            "agent": "results_interpreter",
            "query": "Which is the best injector in terms of utility?",
        },
        {
            "id": "query2_3",
            "agent": "results_interpreter",
            "query": "Which producer is best supported?",
        },
        {
            "id": "query2_4",
            "agent": "results_interpreter",
            "query": "Summarize the quality of the models.",
        },
        {
            "id": "query2_5",
            "agent": "results_interpreter",
            "query": "Is there any evidence of thief zones?",
        },
        {
            "id": "query2_6",
            "agent": "results_interpreter",
            "query": "Plot the observed and simulated liquid rates for all wells.",
        },
        {
            "id": "query2_7",
            "agent": "results_interpreter",
            "query": "Is there any evidence of aquifer support?",
        },
        {
            "id": "query2_8",
            "agent": "results_interpreter",
            "query": "Did we use BHP in the simulation?",
        },
        {
            "id": "query2_9",
            "agent": "results_interpreter",
            "query": "Rank the injectors based on their utility.",
        },
        {
            "id": "query2_10",
            "agent": "results_interpreter",
            "query": (
                "For every producer, select three or four points from its production "
                "history and plot observed versus simulated liquid production at those points."
            ),
        },
        {
            "id": "query2_11",
            "agent": "results_interpreter",
            "query": "Give me a summary of the results, focusing on support and channeling.",
        },
        {
            "id": "query2_12",
            "agent": "results_interpreter",
            "query": "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?",
        },
        {
            "id": "queryb_1",
            "agent": ["input_data_analysis", "results_interpreter"],
            "query": "Which 10 producers have the highest oil-to-water ratio, and which injectors support them, if any?",
        },
        {
            "id": "query3_1",
            "agent": "scenario_analyst",
            "query": "I need to shut in well AB. Analyze the consequences and give me concrete actions.",
        },
        {
            "id": "query3_2",
            "agent": "results_interpreter",
            "query": "Summarize the results.",
        },
        {
            "id": "query3_3",
            "agent": "results_interpreter",
            "query": "Is there any evidence of channeling?",
        },
        {
            "id": "query3_4",
            "agent": "scenario_analyst",
            "query": "What will happen if I reduce the injection rate in well AB by 30%?",
        },
        {
            "id": "query3_5",
            "agent": "scenario_analyst",
            "query": "If injector I12 is shut in, which producers are most likely to be affected and which other injectors could compensate for the lost support?",
        },
        {
            "id": "query4_1",
            "agent": "opportunity_scanner",
            "query": "How can I optimize my waterflood? Give me concrete actions.",
        },
        {
            "id": "query4_2",
            "agent": "opportunity_scanner",
            "query": "I need to reduce water production. Give me concrete actions.",
        },
        {
            "id": "query4_3",
            "agent": "opportunity_scanner",
            "query": "Identify injectors where injection could potentially be reduced with the lowest expected impact on supported oil production.",
        },
    ]

    import pandas as pd
    pd.set_option("display.max_colwidth", None)

    results = []
    response = None 

    if q is None:
        questions = planner_golden_questions[7:9]
    else:
        questions = [{'id':'single_question','agent':'tbd','query':q}]

    for question in questions:
        print(f"Running test for question: {question['query']}")

        query = question['query']

        try:
            response = agent.invoke({
                    "messages": [
                        {"role": "user", "content": query}
                    ]
                })['structured_response']

            #plan: ExecutionPlanTemplate = response['structured_response']
                
            print(" Successfully generated a bulletproof, deadlock-free plan!")
            print(f"Goal: {response.user_intent}")


            agent_names = ",".join([task.agent for task in response.tasks])

            results.append({
                
                "id": question['id'],
                "query": query,
                "expected_agent": question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent']),
                "actual_agent": agent_names,
                "user_intent": response.user_intent,
                "success": agent_names == (question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent'])),
                #"tasks": [task.dict() for task in response.tasks]
            })

        except ValidationError as e:
            print("❌ LLM Plan failed validation logic!")
            # e.errors() will contain your specific "Circular Dependency" or "Missing ID" text
            print(e) 
            
        except Exception as e:
            print(f"An unexpected framework or API error occurred: {e}")

    df = pd.DataFrame( results )
    df = df[ ['success'] + list(df.columns[0:-1]) ] 
    display( df ) 

    if q is None:
        return df  
    else:
        return df, response 


In [32]:
from agentic_system.global_execution.global_execution_prompt import global_planner_prompt as global_planner_prompt
from agentic_system.common.base_plan import ExecutionPlanTemplate



In [33]:

llm = get_llm() 


agent = create_agent(
            model=llm,
            system_prompt = global_planner_prompt,
            #tools=self.tools,
            response_format=ToolStrategy(ExecutionPlanTemplate)
        )

q="""
Which sector had the largest increase in water injection over the last two years, 
anlalyze producer support and injector efficiency in that sector and tell me 
how was the history match in that sector?
"""
df, response = test_planner(global_planner_prompt, agent, q)


zero temp, seed 42, top_p = 1
Running test for question: 
Which sector had the largest increase in water injection over the last two years, 
anlalyze producer support and injector efficiency in that sector and tell me 
how was the history match in that sector?

 Successfully generated a bulletproof, deadlock-free plan!
Goal: Analyze the sector with the largest increase in water injection over the last two years, focusing on producer support, injector efficiency, and history match quality.


,success,id,query,expected_agent,actual_agent,user_intent
0,False,single_question,"\nWhich sector had the largest increase in water injection over the last two years, \nanlalyze producer support and injector efficiency in that sector and tell me \nhow was the history match in that sector?\n",tbd,"input_data_analysis,results_interpreter,results_interpreter","Analyze the sector with the largest increase in water injection over the last two years, focusing on producer support, injector efficiency, and history match quality."


In [35]:
assert response is not None
response.execution_order
    


[ExecutionTask(instruction='Identify the sector with the largest increase in water injection over the last two years.', agent='input_data_analysis', task_id='identify_sector_largest_injection_increase', depends_on=None),
 ExecutionTask(instruction='Analyze producer support and injector efficiency in the sector identified by the preceding task.', agent='results_interpreter', task_id='analyze_support_efficiency', depends_on=['identify_sector_largest_injection_increase']),
 ExecutionTask(instruction='Evaluate the history match quality in the sector identified by the first task.', agent='results_interpreter', task_id='evaluate_history_match', depends_on=['identify_sector_largest_injection_increase'])]

# Using the base classes

## Factory 

In [10]:
from agentic_system.common.base_task import BaseSystemTask
from agentic_system.common.base_plan import PlannerComponent,PlannerConfig

from agentic_system.global_execution.global_execution_prompt import global_planner_prompt as global_planner_prompt
from agentic_system.common.base_plan import ExecutionPlanTemplate



In [14]:
config = PlannerConfig( prompt=global_planner_prompt)#, response_format=ToolStrategy(ExecutionPlanTemplate) )

llm = get_llm() 

planner = PlannerComponent( llm=llm, config=config, plan_model=ExecutionPlanTemplate )


q="""
Which sector had the largest increase in water injection over the last two years, 
anlalyze producer support and injector efficiency in that sector and tell me 
how was the history match in that sector?
"""
r = planner.run( q )


zero temp, seed 42, top_p = 1


In [16]:
r.model_dump() 

{'user_intent': 'Determine the sector with the largest increase in water injection over the last two years, analyze producer support and injector efficiency in that sector, and evaluate the history match quality for that sector.',
 'tasks': [{'instruction': 'Identify the sector with the largest increase in water injection over the last two years.',
   'agent': 'input_data_analysis',
   'task_id': 'identify_sector_injection_increase',
   'depends_on': None},
  {'instruction': 'Analyze the producer support and injector efficiency in the sector identified by the previous task.',
   'agent': 'results_interpreter',
   'task_id': 'analyze_support_efficiency',
   'depends_on': ['identify_sector_injection_increase']},
  {'instruction': 'Evaluate the history match quality in the sector identified by the first task.',
   'agent': 'results_interpreter',
   'task_id': 'evaluate_history_match',
   'depends_on': ['identify_sector_injection_increase']}]}

In [ ]:

def test_planner( planner ):
    import pandas as pd
    pd.set_option("display.max_colwidth", None)

    # ``agent`` is a string for the usual single-agent route and a list only when
    # the request genuinely requires more than one specialist.
    planner_golden_questions = [
        {
            "id": "query0_0",
            "agent": "direct_answer",
            "query": "Why can a high water cut be a problem even when total liquid production remains high?",
        },
        {
            "id": "query0_1",
            "agent": "results_interpreter",
            "query": "Which wells are performing badly?",
        },
        {
            "id": "query1_1",
            "agent": "input_data_analysis",
            "query": "plot the liquid injection rates in sector 1 over time",
        },
        {
            "id": "query1_2",
            "agent": "input_data_analysis",
            "query": "list the 3 wells with the highest watercut",
        },
        {
            "id": "query1_3",
            "agent": "input_data_analysis",
            "query": "plot the VRR split by sector",
        },
        {
            "id": "query1_4",
            "agent": "input_data_analysis",
            "query": "Define VRR and plot the VRR over time for sectors 1, 2, and 3",
        },
        {
            "id": "query1_5",
            "agent": "input_data_analysis",
            "query": "Is the data frequency daily or monthly?",
        },
        {
            "id": "query1_6",
            "agent": "input_data_analysis",
            "query": "Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",
        },
        {
            "id": "query2_1",
            "agent": "results_interpreter",
            "query": "Why weren't 12 of the initially screened wells modelled?",
        },
        {
            "id": "query2_2",
            "agent": "results_interpreter",
            "query": "Which is the best injector in terms of utility?",
        },
        {
            "id": "query2_3",
            "agent": "results_interpreter",
            "query": "Which producer is best supported?",
        },
        {
            "id": "query2_4",
            "agent": "results_interpreter",
            "query": "Summarize the quality of the models.",
        },
        {
            "id": "query2_5",
            "agent": "results_interpreter",
            "query": "Is there any evidence of thief zones?",
        },
        {
            "id": "query2_6",
            "agent": "results_interpreter",
            "query": "Plot the observed and simulated liquid rates for all wells.",
        },
        {
            "id": "query2_7",
            "agent": "results_interpreter",
            "query": "Is there any evidence of aquifer support?",
        },
        {
            "id": "query2_8",
            "agent": "results_interpreter",
            "query": "Did we use BHP in the simulation?",
        },
        {
            "id": "query2_9",
            "agent": "results_interpreter",
            "query": "Rank the injectors based on their utility.",
        },
        {
            "id": "query2_10",
            "agent": "results_interpreter",
            "query": (
                "For every producer, select three or four points from its production "
                "history and plot observed versus simulated liquid production at those points."
            ),
        },
        {
            "id": "query2_11",
            "agent": "results_interpreter",
            "query": "Give me a summary of the results, focusing on support and channeling.",
        },
        {
            "id": "query2_12",
            "agent": "results_interpreter",
            "query": "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?",
        },
        {
            "id": "queryb_1",
            "agent": ["input_data_analysis", "results_interpreter"],
            "query": "Which 10 producers have the highest oil-to-water ratio, and which injectors support them, if any?",
        },
        {
            "id": "query3_1",
            "agent": "scenario_analyst",
            "query": "I need to shut in well AB. Analyze the consequences and give me concrete actions.",
        },
        {
            "id": "query3_2",
            "agent": "results_interpreter",
            "query": "Summarize the results.",
        },
        {
            "id": "query3_3",
            "agent": "results_interpreter",
            "query": "Is there any evidence of channeling?",
        },
        {
            "id": "query3_4",
            "agent": "scenario_analyst",
            "query": "What will happen if I reduce the injection rate in well AB by 30%?",
        },
        {
            "id": "query3_5",
            "agent": "scenario_analyst",
            "query": "If injector I12 is shut in, which producers are most likely to be affected and which other injectors could compensate for the lost support?",
        },
        {
            "id": "query4_1",
            "agent": "opportunity_scanner",
            "query": "How can I optimize my waterflood? Give me concrete actions.",
        },
        {
            "id": "query4_2",
            "agent": "opportunity_scanner",
            "query": "I need to reduce water production. Give me concrete actions.",
        },
        {
            "id": "query4_3",
            "agent": "opportunity_scanner",
            "query": "Identify injectors where injection could potentially be reduced with the lowest expected impact on supported oil production.",
        },
    ]


    results = []
    for question in planner_golden_questions:#[6:10]:
        print(f"Running test for question: {question['query']}")

        query = question['query']
        #response = agent.invoke({
        #        "messages": [
        #            {"role": "user", "content": query}
        #        ]
        #    })['structured_response']
        response = planner.run( query )

        agent_names = ",".join([task.agent for task in response.tasks])

        results.append({
            
            "id": question['id'],
            "query": query,
            "expected_agent": question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent']),
            "actual_agent": agent_names,
            "user_intent": response.user_intent,
            "success": agent_names == (question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent'])),
            #"tasks": [task.dict() for task in response.tasks]
        })


    df = pd.DataFrame( results )
    df = df[ ['success'] + list(df.columns[0:-1]) ] 
    display( df ) 
    return df 

df = test_planner(planner)



Running test for question: Is the data frequency daily or monthly?
Running test for question: Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?
Running test for question: Why weren't 12 of the initially screened wells modelled?
Running test for question: Which is the best injector in terms of utility?


,success,id,query,expected_agent,actual_agent,user_intent
0,True,query1_5,Is the data frequency daily or monthly?,input_data_analysis,input_data_analysis,Determine the frequency of the input data in the project.
1,True,query1_6,"Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",input_data_analysis,input_data_analysis,Determine the sector with the largest increase in water injection over the last two years and analyze its oil production trend during the same period.
2,True,query2_1,Why weren't 12 of the initially screened wells modelled?,results_interpreter,results_interpreter,Understand why 12 initially screened wells were not included in the model.
3,True,query2_2,Which is the best injector in terms of utility?,results_interpreter,results_interpreter,Determine the injector with the highest utility based on simulation results.


## Type lock

In [64]:
planner = PlannerComponent( llm=llm, config=config, plan_model=ExecutionPlanTemplate )
df = test_planner(planner)


Running test for question: Is the data frequency daily or monthly?
Running test for question: Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?
Running test for question: Why weren't 12 of the initially screened wells modelled?
Running test for question: Which is the best injector in terms of utility?


,success,id,query,expected_agent,actual_agent,user_intent
0,True,query1_5,Is the data frequency daily or monthly?,input_data_analysis,input_data_analysis,Determine the frequency of the input data in the project.
1,True,query1_6,"Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",input_data_analysis,input_data_analysis,Determine the sector with the largest increase in water injection over the last two years and analyze its oil production trend during the same period.
2,True,query2_1,Why weren't 12 of the initially screened wells modelled?,results_interpreter,results_interpreter,Understand why 12 initially screened wells were not included in the model.
3,True,query2_2,Which is the best injector in terms of utility?,results_interpreter,results_interpreter,Determine the injector with the highest utility based on simulation results.


# Version 0.0 planner 



In [8]:

from dataclasses import dataclass
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')
from agentic_system.common.base_task import BaseSystemTask
from agentic_system.common.base_plan import PlannerComponent,PlannerConfig, ExecutionPlanTemplate

from agentic_system.global_execution.global_execution_prompt import router_prompt
from get_llm_model import azure_llm_if as get_llm 



def test_planner( planner ):
    import pandas as pd
    pd.set_option("display.max_colwidth", None)

    # ``agent`` is a string for the usual single-agent route and a list only when
    # the request genuinely requires more than one specialist.
    planner_golden_questions = [
        {
            "id": "query0_0",
            "agent": "direct_answer",
            "query": "Why can a high water cut be a problem even when total liquid production remains high?",
        },
        {
            "id": "query0_1",
            "agent": "results_interpreter",
            "query": "Which wells are performing badly?",
        },
        {
            "id": "query1_1",
            "agent": "input_data_analysis",
            "query": "plot the liquid injection rates in sector 1 over time",
        },
        {
            "id": "query1_2",
            "agent": "input_data_analysis",
            "query": "list the 3 wells with the highest watercut",
        },
        {
            "id": "query1_3",
            "agent": "input_data_analysis",
            "query": "plot the VRR split by sector",
        },
        {
            "id": "query1_4",
            "agent": "input_data_analysis",
            "query": "Define VRR and plot the VRR over time for sectors 1, 2, and 3",
        },
        {
            "id": "query1_5",
            "agent": "input_data_analysis",
            "query": "Is the data frequency daily or monthly?",
        },
        {
            "id": "query1_6",
            "agent": "input_data_analysis",
            "query": "Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",
        },
        {
            "id": "query2_1",
            "agent": "results_interpreter",
            "query": "Why weren't 12 of the initially screened wells modelled?",
        },
        {
            "id": "query2_2",
            "agent": "results_interpreter",
            "query": "Which is the best injector in terms of utility?",
        },
                {
            "id": "query2_21",
            "agent": "results_interpreter",
            "query": "Which is the best injector?",
        },
        {
            "id": "query2_3",
            "agent": "results_interpreter",
            "query": "Which producer is best supported?",
        },
        {
            "id": "query2_4",
            "agent": "results_interpreter",
            "query": "Summarize the quality of the models.",
        },
        {
            "id": "query2_5",
            "agent": "results_interpreter",
            "query": "Is there any evidence of thief zones?",
        },
        {
            "id": "query2_6",
            "agent": "results_interpreter",
            "query": "Plot the observed and simulated liquid rates for all wells.",
        },
        {
            "id": "query2_7",
            "agent": "results_interpreter",
            "query": "Is there any evidence of aquifer support?",
        },
        {
            "id": "query2_8",
            "agent": "results_interpreter",
            "query": "Did we use BHP in the simulation?",
        },
        {
            "id": "query2_9",
            "agent": "results_interpreter",
            "query": "Rank the injectors based on their utility.",
        },
        {
            "id": "query2_10",
            "agent": "results_interpreter",
            "query": (
                "For every producer, select three or four points from its production "
                "history and plot observed versus simulated liquid production at those points."
            ),
        },
        {
            "id": "query2_11",
            "agent": "results_interpreter",
            "query": "Give me a summary of the results, focusing on support and channeling.",
        },
        {
            "id": "query2_12",
            "agent": "results_interpreter",
            "query": "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?",
        },
        {
            "id": "queryb_1",
            "agent": ["input_data_analysis", "results_interpreter"],
            "query": "Which 10 producers have the highest oil-to-water ratio, and which injectors support them, if any?",
        },
        {
            "id": "query3_1",
            "agent": "scenario_analyst",
            "query": "I need to shut in well AB. Analyze the consequences and give me concrete actions.",
        },
        {
            "id": "query3_2",
            "agent": "results_interpreter",
            "query": "Summarize the results.",
        },
        {
            "id": "query3_3",
            "agent": "results_interpreter",
            "query": "Is there any evidence of channeling?",
        },
        {
            "id": "query3_4",
            "agent": "scenario_analyst",
            "query": "What will happen if I reduce the injection rate in well AB by 30%?",
        },
        {
            "id": "query3_5",
            "agent": "scenario_analyst",
            "query": "If injector I12 is shut in, which producers are most likely to be affected and which other injectors could compensate for the lost support?",
        },
        {
            "id": "query4_1",
            "agent": "opportunity_scanner",
            "query": "How can I optimize my waterflood? Give me concrete actions.",
        },
        {
            "id": "query4_2",
            "agent": "opportunity_scanner",
            "query": "I need to reduce water production. Give me concrete actions.",
        },
        {
            "id": "query4_3",
            "agent": "opportunity_scanner",
            "query": "Identify injectors where injection could potentially be reduced with the lowest expected impact on supported oil production.",
        },
    ]


    results = []
    for n,question in enumerate(planner_golden_questions):#[6:10]:
        print(f"Running test for question [{n}]: {question['query']}")

        query = question['query']
        #response = agent.invoke({
        #        "messages": [
        #            {"role": "user", "content": query}
        #        ]
        #    })['structured_response']
        response = planner.run( query )

        agent_names = ",".join([task.agent for task in response.tasks])

        results.append({
            
            "id": question['id'],
            "query": query,
            "expected_agent": question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent']),
            "actual_agent": agent_names,
            "user_intent": response.user_intent,
            "success": agent_names == (question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent'])),
            #"tasks": [task.dict() for task in response.tasks]
        })


    df = pd.DataFrame( results )
    df = df[ ['success'] + list(df.columns[0:-1]) ] 
    display( df ) 
    return df 

llm = get_llm() 

config  = PlannerConfig( prompt=router_prompt)#, response_format=ToolStrategy(ExecutionPlanTemplate) )
planner = PlannerComponent( llm=llm, config=config, plan_model=ExecutionPlanTemplate )
df = test_planner(planner)



imported
zero temp, seed 42, top_p = 1
Running test for question [0]: Why can a high water cut be a problem even when total liquid production remains high?
Running test for question [1]: Which wells are performing badly?
Running test for question [2]: plot the liquid injection rates in sector 1 over time
Running test for question [3]: list the 3 wells with the highest watercut
Running test for question [4]: plot the VRR split by sector
Running test for question [5]: Define VRR and plot the VRR over time for sectors 1, 2, and 3
Running test for question [6]: Is the data frequency daily or monthly?


KeyboardInterrupt: 

# Final global planner 

Improved prompt for multi-part queries
Structured output 

In [ ]:

from dataclasses import dataclass
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')
from agentic_system.common.base_task import BaseSystemTask
from agentic_system.common.base_plan import PlannerComponent,PlannerConfig, ExecutionPlanTemplate

from agentic_system.global_execution.global_execution_prompt import global_planner_prompt
from get_llm_model import azure_llm_if as get_llm 



def test_planner( planner ):
    import pandas as pd
    pd.set_option("display.max_colwidth", None)

    # ``agent`` is a string for the usual single-agent route and a list only when
    # the request genuinely requires more than one specialist.
    planner_golden_questions = [
        {
            "id": "query0_0",
            "agent": "direct_answer",
            "query": "Why can a high water cut be a problem even when total liquid production remains high?",
        },
        {
            "id": "query0_1",
            "agent": "results_interpreter",
            "query": "Which wells are performing badly?",
        },
        {
            "id": "query1_1",
            "agent": "input_data_analysis",
            "query": "plot the liquid injection rates in sector 1 over time",
        },
        {
            "id": "query1_2",
            "agent": "input_data_analysis",
            "query": "list the 3 wells with the highest watercut",
        },
        {
            "id": "query1_3",
            "agent": "input_data_analysis",
            "query": "plot the VRR split by sector",
        },
        {
            "id": "query1_4",
            "agent": "direct_answer,input_data_analysis",
            "query": "Define VRR and plot the VRR over time for sectors 1, 2, and 3",
        },
        {
            "id": "query1_5",
            "agent": "input_data_analysis",
            "query": "Is the data frequency daily or monthly?",
        },
        {
            "id": "query1_6",
            "agent": "input_data_analysis",
            "query": "Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",
        },
        {
            "id": "query2_1",
            "agent": "results_interpreter",
            "query": "Why weren't 12 of the initially screened wells modelled?",
        },
        {
            "id": "query2_2",
            "agent": "results_interpreter",
            "query": "Which is the best injector in terms of utility?",
        },
                {
            "id": "query2_21",
            "agent": "results_interpreter",
            "query": "Which is the best injector?",
        },
        {
            "id": "query2_3",
            "agent": "results_interpreter",
            "query": "Which producer is best supported?",
        },
        {
            "id": "query2_4",
            "agent": "results_interpreter",
            "query": "Summarize the quality of the models.",
        },
        {
            "id": "query2_5",
            "agent": "results_interpreter",
            "query": "Is there any evidence of thief zones?",
        },
        {
            "id": "query2_6",
            "agent": "results_interpreter",
            "query": "Plot the observed and simulated liquid rates for all wells.",
        },
        {
            "id": "query2_7",
            "agent": "results_interpreter",
            "query": "Is there any evidence of aquifer support?",
        },
        {
            "id": "query2_8",
            "agent": "results_interpreter",
            "query": "Did we use BHP in the simulation?",
        },
        {
            "id": "query2_9",
            "agent": "results_interpreter",
            "query": "Rank the injectors based on their utility.",
        },
        {
            "id": "query2_10",
            "agent": "results_interpreter",
            "query": (
                "For every producer, select three or four points from its production "
                "history and plot observed versus simulated liquid production at those points."
            ),
        },
        {
            "id": "query2_11",
            "agent": "results_interpreter",
            "query": "Give me a summary of the results, focusing on support and channeling.",
        },
        {
            "id": "query2_12",
            "agent": "results_interpreter",
            "query": "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?",
        },
        {
            "id": "queryb_1",
            "agent": ["input_data_analysis", "results_interpreter"],
            "query": "Which 10 producers have the highest oil-to-water ratio, and which injectors support them, if any?",
        },
        {
            "id": "query3_1",
            "agent": "scenario_analyst",
            "query": "I need to shut in well AB. Analyze the consequences and give me concrete actions.",
        },
        {
            "id": "query3_2",
            "agent": "results_interpreter",
            "query": "Summarize the results.",
        },
        {
            "id": "query3_3",
            "agent": "results_interpreter",
            "query": "Is there any evidence of channeling?",
        },
        {
            "id": "query3_4",
            "agent": "scenario_analyst",
            "query": "What will happen if I reduce the injection rate in well AB by 30%?",
        },
        {
            "id": "query3_5",
            "agent": "scenario_analyst",
            "query": "If injector I12 is shut in, which producers are most likely to be affected and which other injectors could compensate for the lost support?",
        },
        {
            "id": "query4_1",
            "agent": "opportunity_scanner",
            "query": "How can I optimize my waterflood? Give me concrete actions.",
        },
        {
            "id": "query4_2",
            "agent": "opportunity_scanner",
            "query": "I need to reduce water production. Give me concrete actions.",
        },
        {
            "id": "query4_3",
            "agent": "opportunity_scanner",
            "query": "Identify injectors where injection could potentially be reduced with the lowest expected impact on supported oil production.",
        },
    ]


    results = []
    for n,question in enumerate(planner_golden_questions):#[6:10]:
        print(f"Running test for question [{n}]: {question['query']}")

        query = question['query']
        #response = agent.invoke({
        #        "messages": [
        #            {"role": "user", "content": query}
        #        ]
        #    })['structured_response']
        response = planner.run( query )

        agent_names = ",".join([task.agent for task in response.tasks])

        results.append({
            
            "id": question['id'],
            "query": query,
            "expected_agent": question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent']),
            "actual_agent": agent_names,
            "user_intent": response.user_intent,
            "success": agent_names == (question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent'])),
            #"tasks": [task.dict() for task in response.tasks]
        })


    df = pd.DataFrame( results )
    df = df[ ['success'] + list(df.columns[0:-1]) ] 
    display( df ) 
    return df 

llm = get_llm() 

config  = PlannerConfig( prompt=global_planner_prompt)#, response_format=ToolStrategy(ExecutionPlanTemplate) )
planner = PlannerComponent( llm=llm, config=config, plan_model=ExecutionPlanTemplate )
df = test_planner(planner)

imported
zero temp, seed 42, top_p = 1
Running test for question [0]: Why can a high water cut be a problem even when total liquid production remains high?
Running test for question [1]: Which wells are performing badly?
Running test for question [2]: plot the liquid injection rates in sector 1 over time
Running test for question [3]: list the 3 wells with the highest watercut
Running test for question [4]: plot the VRR split by sector
Running test for question [5]: Define VRR and plot the VRR over time for sectors 1, 2, and 3
Running test for question [6]: Is the data frequency daily or monthly?
Running test for question [7]: Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?
Running test for question [8]: Why weren't 12 of the initially screened wells modelled?
Running test for question [9]: Which is the best injector in terms of utility?
Running test for question [10]: Which is the best injector

,success,id,query,expected_agent,actual_agent,user_intent
0,True,query0_0,Why can a high water cut be a problem even when total liquid production remains high?,direct_answer,direct_answer,Explain why a high water cut can be problematic despite high total liquid production.
1,True,query0_1,Which wells are performing badly?,results_interpreter,results_interpreter,Identify wells that are underperforming.
2,True,query1_1,plot the liquid injection rates in sector 1 over time,input_data_analysis,input_data_analysis,Visualize the liquid injection rates in sector 1 over time.
3,True,query1_2,list the 3 wells with the highest watercut,input_data_analysis,input_data_analysis,Identify the three wells with the highest watercut.
4,True,query1_3,plot the VRR split by sector,input_data_analysis,input_data_analysis,Visualize the VRR split by sector.
5,False,query1_4,"Define VRR and plot the VRR over time for sectors 1, 2, and 3",input_data_analysis,"direct_answer,input_data_analysis",Understand the concept of VRR and analyze its historical trend for specific sectors.
6,True,query1_5,Is the data frequency daily or monthly?,input_data_analysis,input_data_analysis,Determine the frequency of the data in the project.
7,True,query1_6,"Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",input_data_analysis,input_data_analysis,Identify the sector with the largest increase in water injection over the last two years and analyze its oil production trend during the same period.
8,True,query2_1,Why weren't 12 of the initially screened wells modelled?,results_interpreter,results_interpreter,Understand why 12 initially screened wells were not included in the final model.
9,True,query2_2,Which is the best injector in terms of utility?,results_interpreter,results_interpreter,Determine the injector with the highest utility based on simulation results.


In [2]:
df

,success,id,query,expected_agent,actual_agent,user_intent
0,True,query0_0,Why can a high water cut be a problem even when total liquid production remains high?,direct_answer,direct_answer,Explain why a high water cut can be problematic despite high total liquid production.
1,True,query0_1,Which wells are performing badly?,results_interpreter,results_interpreter,Identify wells that are underperforming.
2,True,query1_1,plot the liquid injection rates in sector 1 over time,input_data_analysis,input_data_analysis,Visualize the liquid injection rates in sector 1 over time.
3,True,query1_2,list the 3 wells with the highest watercut,input_data_analysis,input_data_analysis,Identify the three wells with the highest watercut.
4,True,query1_3,plot the VRR split by sector,input_data_analysis,input_data_analysis,Visualize the VRR split by sector.
5,False,query1_4,"Define VRR and plot the VRR over time for sectors 1, 2, and 3",input_data_analysis,"direct_answer,input_data_analysis",Understand the concept of VRR and analyze its historical trend for specific sectors.
6,True,query1_5,Is the data frequency daily or monthly?,input_data_analysis,input_data_analysis,Determine the frequency of the data in the project.
7,True,query1_6,"Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",input_data_analysis,input_data_analysis,Identify the sector with the largest increase in water injection over the last two years and analyze its oil production trend during the same period.
8,True,query2_1,Why weren't 12 of the initially screened wells modelled?,results_interpreter,results_interpreter,Understand why 12 initially screened wells were not included in the final model.
9,True,query2_2,Which is the best injector in terms of utility?,results_interpreter,results_interpreter,Determine the injector with the highest utility based on simulation results.


In [3]:
query = "If I were to shut one injector, which one would be the one that would affect the less possible the oil production? "

response = planner.run( query )



In [4]:
response

ExecutionPlanTemplate(user_intent='Determine which injector can be shut down with minimal impact on oil production.', tasks=[ExecutionTask(instruction='Identify the injector whose shutdown would have the least impact on oil production.', agent='scenario_analyst', task_id='evaluate_injector_shutdown', depends_on=None)])

In [ ]:
COCEPTS:

Producer well performance: For a producer, performance is measured as the ratio of its oil production to its water production. 
A high water cut indicates that a well is producing a large amount of water relative to oil, which can be problematic 
for several reasons, including increased operational costs and reduced efficiency. 

Injector well performance: For an injector, performance is measured as the sum of the gains over all producers connected to the injector.
An injector that performs well, supports one or more producers with a cummulative gain greater than 1. 
However, an injector might be performant but at the same time it can be of low utility. This is because the injector utility 
depends on the amount of oil produced as a direct consequence of injection. 
Performance and utility are related but distinct concepts. For injectors, a more useful measure is "utiity" 

Producer support:

Producer allocation:


Injector connectivity: An injector is considered to support a producer if it is connected to that producer in the reservoir model.




In [ ]:
query="Which 10 producers have the highest oil-to-water ratio, and which injectors support them, if any"


config  = PlannerConfig( prompt=router_prompt)#, response_format=ToolStrategy(ExecutionPlanTemplate) )
planner = PlannerComponent( llm=llm, config=config, plan_model=ExecutionPlanTemplate )



config  = PlannerConfig( prompt=prompt)#, response_format=ToolStrategy(ExecutionPlanTemplate) )
planner = PlannerComponent( llm=llm, config=config, plan_model=ExecutionPlanTemplate )
r = planner.run( query )




NameError: name 'Field' is not defined

In [116]:
for t in r.tasks:
    print()
    print(f"Task assigned to {t.agent}: {t.instruction}")
    print(f"Task ID: {t.task_id}")
    if t.depends_on:
        print(f"Depends on: {', '.join(t.depends_on)}") 

    


Task assigned to input_data_analysis: Identify the 10 producers with the highest oil-to-water ratio based on historical data.
Task ID: rank_producers_by_owr

Task assigned to results_interpreter: Determine which injectors support the producers identified by the preceding task based on simulation results.
Task ID: identify_supporting_injectors
Depends on: rank_producers_by_owr


In [108]:
test_planner(planner)

Running test for question [0]: Why can a high water cut be a problem even when total liquid production remains high?
Running test for question [1]: Which wells are performing badly?
Running test for question [2]: plot the liquid injection rates in sector 1 over time
Running test for question [3]: list the 3 wells with the highest watercut
Running test for question [4]: plot the VRR split by sector
Running test for question [5]: Define VRR and plot the VRR over time for sectors 1, 2, and 3
Running test for question [6]: Is the data frequency daily or monthly?
Running test for question [7]: Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?
Running test for question [8]: Why weren't 12 of the initially screened wells modelled?
Running test for question [9]: Which is the best injector in terms of utility?
Running test for question [10]: Which is the best injector?
Running test for question [11]: Which

,success,id,query,expected_agent,actual_agent,user_intent
0,True,query0_0,Why can a high water cut be a problem even when total liquid production remains high?,direct_answer,direct_answer,Explain why a high water cut can be problematic despite high total liquid production.
1,True,query0_1,Which wells are performing badly?,results_interpreter,results_interpreter,Identify wells that are underperforming.
2,True,query1_1,plot the liquid injection rates in sector 1 over time,input_data_analysis,input_data_analysis,Visualize the liquid injection rates in sector 1 over time.
3,True,query1_2,list the 3 wells with the highest watercut,input_data_analysis,input_data_analysis,Identify the three wells with the highest watercut.
4,True,query1_3,plot the VRR split by sector,input_data_analysis,input_data_analysis,To visualize the VRR (Voidage Replacement Ratio) for each sector in the project.
5,False,query1_4,"Define VRR and plot the VRR over time for sectors 1, 2, and 3",input_data_analysis,"direct_answer,input_data_analysis",Define VRR and analyze its trend over time for specific sectors.
6,True,query1_5,Is the data frequency daily or monthly?,input_data_analysis,input_data_analysis,Determine the frequency of the data in the project.
7,True,query1_6,"Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",input_data_analysis,input_data_analysis,Determine the sector with the largest increase in water injection over the last two years and analyze its oil production trend during the same period.
8,True,query2_1,Why weren't 12 of the initially screened wells modelled?,results_interpreter,results_interpreter,Understand why 12 initially screened wells were not included in the model.
9,True,query2_2,Which is the best injector in terms of utility?,results_interpreter,results_interpreter,Determine the injector with the highest utility based on simulation results.


,success,id,query,expected_agent,actual_agent,user_intent
0,True,query0_0,Why can a high water cut be a problem even when total liquid production remains high?,direct_answer,direct_answer,Explain why a high water cut can be problematic despite high total liquid production.
1,True,query0_1,Which wells are performing badly?,results_interpreter,results_interpreter,Identify wells that are underperforming.
2,True,query1_1,plot the liquid injection rates in sector 1 over time,input_data_analysis,input_data_analysis,Visualize the liquid injection rates in sector 1 over time.
3,True,query1_2,list the 3 wells with the highest watercut,input_data_analysis,input_data_analysis,Identify the three wells with the highest watercut.
4,True,query1_3,plot the VRR split by sector,input_data_analysis,input_data_analysis,To visualize the VRR (Voidage Replacement Ratio) for each sector in the project.
5,False,query1_4,"Define VRR and plot the VRR over time for sectors 1, 2, and 3",input_data_analysis,"direct_answer,input_data_analysis",Define VRR and analyze its trend over time for specific sectors.
6,True,query1_5,Is the data frequency daily or monthly?,input_data_analysis,input_data_analysis,Determine the frequency of the data in the project.
7,True,query1_6,"Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",input_data_analysis,input_data_analysis,Determine the sector with the largest increase in water injection over the last two years and analyze its oil production trend during the same period.
8,True,query2_1,Why weren't 12 of the initially screened wells modelled?,results_interpreter,results_interpreter,Understand why 12 initially screened wells were not included in the model.
9,True,query2_2,Which is the best injector in terms of utility?,results_interpreter,results_interpreter,Determine the injector with the highest utility based on simulation results.


# Final planner


In [1]:

from dataclasses import dataclass
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')


import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if as get_llm 


from agentic_system.common.base_task import BaseSystemTask, ExecutionTask
from agentic_system.common.base_plan import PlannerComponent,PlannerConfig, ExecutionPlanTemplate
from agentic_system.global_execution.global_execution_prompt import global_planner_prompt



imported


In [ ]:

query="Which 10 producers have the highest oil-to-water ratio, and which injectors support them, if any"


In [4]:

llm = get_llm() 

  
config  = PlannerConfig( prompt=global_planner_prompt)#, response_format=ToolStrategy(ExecutionPlanTemplate) )
planner = PlannerComponent( llm=llm, config=config, plan_model=ExecutionPlanTemplate )
r = planner.run( query )


zero temp, seed 42, top_p = 1


In [5]:
for t in r.tasks:
    print()
    print(f"Task assigned to {t.agent}: {t.instruction}")
    print(f"Task ID: {t.task_id}")
    if t.depends_on:
        print(f"Depends on: {', '.join(t.depends_on)}") 



Task assigned to input_data_analysis: Identify the 10 producers with the highest oil-to-water ratio based on historical data.
Task ID: identify_top_producers

Task assigned to results_interpreter: Determine which injectors support the producers identified by the preceding task based on simulation results.
Task ID: determine_supporting_injectors
Depends on: identify_top_producers


## Testing. Golden queries 

In [ ]:
def test_planner( planner ):
    import pandas as pd
    import time
    pd.set_option("display.max_colwidth", None)

    # ``agent`` is a string for the usual single-agent route and a list only when
    # the request genuinely requires more than one specialist.
    planner_golden_questions = [
        {
            "id": "query0_0",
            "agent": "direct_answer",
            "query": "Why can a high water cut be a problem even when total liquid production remains high?",
        },
        {
            "id": "query0_1",
            "agent": "results_interpreter",
            "query": "Which wells are performing badly?",
        },
        {
            "id": "query1_1",
            "agent": "input_data_analysis",
            "query": "plot the liquid injection rates in sector 1 over time",
        },
        {
            "id": "query1_2",
            "agent": "input_data_analysis",
            "query": "list the 3 wells with the highest watercut",
        },
        {
            "id": "query1_3",
            "agent": "input_data_analysis",
            "query": "plot the VRR split by sector",
        },
        {
            "id": "query1_4",
            "agent": "input_data_analysis",
            "query": "Define VRR and plot the VRR over time for sectors 1, 2, and 3",
        },
        {
            "id": "query1_5",
            "agent": "input_data_analysis",
            "query": "Is the data frequency daily or monthly?",
        },
        {
            "id": "query1_6",
            "agent": "input_data_analysis",
            "query": "Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",
        },
        {
            "id": "query2_1",
            "agent": "results_interpreter",
            "query": "Why weren't 12 of the initially screened wells modelled?",
        },
        {
            "id": "query2_2",
            "agent": "results_interpreter",
            "query": "Which is the best injector in terms of utility?",
        },
                {
            "id": "query2_21",
            "agent": "results_interpreter",
            "query": "Which is the best injector?",
        },
        {
            "id": "query2_3",
            "agent": "results_interpreter",
            "query": "Which producer is best supported?",
        },
        {
            "id": "query2_4",
            "agent": "results_interpreter",
            "query": "Summarize the quality of the models.",
        },
        {
            "id": "query2_5",
            "agent": "results_interpreter",
            "query": "Is there any evidence of thief zones?",
        },
        {
            "id": "query2_6",
            "agent": "results_interpreter",
            "query": "Plot the observed and simulated liquid rates for all wells.",
        },
        {
            "id": "query2_7",
            "agent": "results_interpreter",
            "query": "Is there any evidence of aquifer support?",
        },
        {
            "id": "query2_8",
            "agent": "results_interpreter",
            "query": "Did we use BHP in the simulation?",
        },
        {
            "id": "query2_9",
            "agent": "results_interpreter",
            "query": "Rank the injectors based on their utility.",
        },
        {
            "id": "query2_10",
            "agent": "results_interpreter",
            "query": (
                "For every producer, select three or four points from its production "
                "history and plot observed versus simulated liquid production at those points."
            ),
        },
        {
            "id": "query2_11",
            "agent": "results_interpreter",
            "query": "Give me a summary of the results, focusing on support and channeling.",
        },
        {
            "id": "query2_12",
            "agent": "results_interpreter",
            "query": "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?",
        },
        {
            "id": "queryb_1",
            "agent": ["input_data_analysis", "results_interpreter"],
            "query": "Which 10 producers have the highest oil-to-water ratio, and which injectors support them, if any?",
        },
        {
            "id": "query3_1",
            "agent": "scenario_analyst",
            "query": "I need to shut in well AB. Analyze the consequences and give me concrete actions.",
        },
        {
            "id": "query3_2",
            "agent": "results_interpreter",
            "query": "Summarize the results.",
        },
        {
            "id": "query3_3",
            "agent": "results_interpreter",
            "query": "Is there any evidence of channeling?",
        },
        {
            "id": "query3_4",
            "agent": "scenario_analyst",
            "query": "What will happen if I reduce the injection rate in well AB by 30%?",
        },
        {
            "id": "query3_5",
            "agent": "scenario_analyst",
            "query": "If injector I12 is shut in, which producers are most likely to be affected and which other injectors could compensate for the lost support?",
        },
        {
            "id": "query4_1",
            "agent": "opportunity_scanner",
            "query": "How can I optimize my waterflood? Give me concrete actions.",
        },
        {
            "id": "query4_2",
            "agent": "opportunity_scanner",
            "query": "I need to reduce water production. Give me concrete actions.",
        },
        {
            "id": "query4_3",
            "agent": "opportunity_scanner",
            "query": "Identify injectors where injection could potentially be reduced with the lowest expected impact on supported oil production.",
        },
    ]


    results = []
    for n,question in enumerate(planner_golden_questions):#[6:10]:
        print(f"Running test for question [{n}]: {question['query']}")

        query = question['query']
        #response = agent.invoke({
        #        "messages": [
        #            {"role": "user", "content": query}
        #        ]
        #    })['structured_response']
        response = planner.run( query )

        agent_names = ",".join([task.agent for task in response.tasks])

        results.append({
            
            "id": question['id'],
            "query": query,
            "expected_agent": question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent']),
            "actual_agent": agent_names,
            "user_intent": response.user_intent,
            "success": agent_names == (question['agent'] if isinstance(question['agent'], str) else ",".join(question['agent'])),
            #"tasks": [task.dict() for task in response.tasks]
        })
        time.sleep(0.1)



    df = pd.DataFrame( results )
    df = df[ ['success'] + list(df.columns[0:-1]) ] 
    display( df ) 
    return df 



In [7]:
test_result = test_planner(planner)

Running test for question [0]: Why can a high water cut be a problem even when total liquid production remains high?
Running test for question [1]: Which wells are performing badly?
Running test for question [2]: plot the liquid injection rates in sector 1 over time
Running test for question [3]: list the 3 wells with the highest watercut
Running test for question [4]: plot the VRR split by sector
Running test for question [5]: Define VRR and plot the VRR over time for sectors 1, 2, and 3
Running test for question [6]: Is the data frequency daily or monthly?
Running test for question [7]: Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?
Running test for question [8]: Why weren't 12 of the initially screened wells modelled?
Running test for question [9]: Which is the best injector in terms of utility?
Running test for question [10]: Which is the best injector?
Running test for question [11]: Which

,success,id,query,expected_agent,actual_agent,user_intent
0,True,query0_0,Why can a high water cut be a problem even when total liquid production remains high?,direct_answer,direct_answer,Explain why a high water cut can be problematic despite high total liquid production.
1,True,query0_1,Which wells are performing badly?,results_interpreter,results_interpreter,Identify wells that are underperforming.
2,True,query1_1,plot the liquid injection rates in sector 1 over time,input_data_analysis,input_data_analysis,Visualize the liquid injection rates in sector 1 over time.
3,True,query1_2,list the 3 wells with the highest watercut,input_data_analysis,input_data_analysis,Identify the three wells with the highest watercut.
4,True,query1_3,plot the VRR split by sector,input_data_analysis,input_data_analysis,Generate a plot of VRR (Voidage Replacement Ratio) split by sector.
5,False,query1_4,"Define VRR and plot the VRR over time for sectors 1, 2, and 3",input_data_analysis,"direct_answer,input_data_analysis",Understand the concept of VRR and analyze its historical trend for specific sectors.
6,True,query1_5,Is the data frequency daily or monthly?,input_data_analysis,input_data_analysis,Determine the frequency of the data in the project.
7,True,query1_6,"Which sector had the largest increase in water injection over the last two years, and did its oil production increase over the same period?",input_data_analysis,input_data_analysis,Determine the sector with the largest increase in water injection over the last two years and analyze its oil production trend during the same period.
8,True,query2_1,Why weren't 12 of the initially screened wells modelled?,results_interpreter,results_interpreter,Understand why 12 initially screened wells were not included in the model.
9,True,query2_2,Which is the best injector in terms of utility?,results_interpreter,results_interpreter,Identify the injector with the highest utility based on simulation results.


In [ ]:

# work today 
# define a CRMResultsObject 
# Storage read + mock from files + read injection table as well  
# Split the connectivity table into 2 tables --part of the object itself-- 
# add some useful methods 
# has_bhp, has_koval, summarize injection, summarize production, connectivity 
# Semantic model for the results 
# Test agent interpreting results with the golden queries 
# Add UI support to send query, check is data/project/simulation changed => reload data in system
# 
#   
# process: 
# Create a project in IF KOC_GenAi_Feature 
# Create a managed folder link to the other managed folder 
# Create a branch 4_1 in the repo 
# Clone the master in KOC_Feature_4_1 and merge the changes from master into 4_1
# Create js4_1 
# Create python/agentic_4_1
#   

# engine:
# export activity
# correct cummulated water injection with activity 
# modify koval for activity 
# 

class CRMResultsObject(dict):

    def has_koval(self):
        # goto the 
        return 'koval' in self and not self['koval'].empty

    def has_watercut(self):
        return self.has_koval()

    def has_bhp(self):
        # goto the 
        return 'bhp' in self and not self['bhp'].empty
 
    def initialize(self,
                   name, 
                   liquid_params, 
                   liquid_rates, 
                   koval_params, 
                   koval_rates,
                   config, 
                   injection_rates,
                   processing_logs, 
                   model_logs):
        pass 
        # load the raw results and perform a split 
        # tailored for non-redundancy and agents 
        # liquid crm is split into connectivity and producer_model tables 


class GenAiUIRequest(BaseModel):
    query: str = Field(
        description="The user's natural language question or request."
    )
    project_id: str = Field(
        description="The unique identifier for the project context."
    )
    simulation_id: Optional[str] = Field(
        default=None,
        description="The unique identifier for the simulation context, if applicable."
    )
    # data filters 


    #selected_wells: Optional[List[str]] = Field(
    #    default=None,
    #    description="A list of well identifiers that the user has selected for focused analysis, if any."
    #)
    #subzone: Optional[str] = Field(
    #    default=None,   
    #    description="The subzone within the project that the user is interested in, if applicable."
    #
    #)


def refresh_data_if_needed_needed( agentic_system,GenAiUIRequest ):
    # check if the project or simulation has changed 
    # if so, reload the data into the agentic_system 
    # this is a placeholder for the actual implementation
    pass


First-pass extra requirements we might need: 

Tool
input data 
pressure_available 
injection_historical_stats
production_historical_stats 

results 
pressure_used 

Input Tables 
inj,prod,loc, distances  

Results tables 
crm_params crm_rates 
koval_params koval_rates 

8 in total. Borderline the easy way of analysts 

We may need to merge results rate for koval and liquid into one table 
Also add injection to the results dataset 
Also add wcut, oil and gas to the results rates dataset. 
 
